In [1]:
import os
import re
import requests as req
from bs4 import BeautifulSoup as bs

In [1]:
pip list


Package                 Version
----------------------- -----------
appnope                 0.1.4
asttokens               3.0.0
attrs                   25.3.0
beautifulsoup4          4.13.4
certifi                 2025.1.31
charset-normalizer      3.4.1
comm                    0.2.2
debugpy                 1.8.11
decorator               5.2.1
exceptiongroup          1.2.2
executing               2.2.0
h11                     0.14.0
idna                    3.10
importlib_metadata      8.6.1
ipykernel               6.29.5
ipython                 9.2.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
jupyter_client          8.6.3
jupyter_core            5.7.2
lxml                    5.4.0
matplotlib-inline       0.1.7
nest_asyncio            1.6.0
outcome                 1.3.0.post0
packaging               25.0
parso                   0.8.4
pexpect                 4.9.0
pickleshare             0.7.5
pip                     25.0
platformdirs            4.3.7
prompt_toolkit     

In [17]:
# 建立資料夾
os.makedirs('project_gutenberg', exist_ok=True)

# 發送HTTP請求
url = 'https://www.gutenberg.org/browse/languages/zh'
response = req.get(url)
response.encoding = 'utf-8'

# 解析HTML內容
soup = bs(response.text, 'html.parser')
links = soup.select('li.pgdbetext > a[href]')

# 定義正則表達式
chinese_regex = re.compile(r'[\u4e00-\u9fff]')

# 初始化計數器
saved_books_count = 0

# 遍歷所有連結
for link in links:
    # 獲取書籍名稱和連結
    book_name = link.get_text()

    # 檢查書名是否包含中文字符
    if not chinese_regex.search(book_name):
        print(f"Skipped (non-Chinese book name): {book_name}")
        continue

    book_url = 'https://www.gutenberg.org' + link['href']

    # 發送請求獲取書籍頁面
    book_response = req.get(book_url)
    book_response.encoding = 'utf-8'

    # 解析書籍頁面
    book_soup = bs(book_response.text, 'html.parser')

    # 嘗試找到書籍下載鏈接
    download_link = book_soup.find('a', href=re.compile(r'.*\.txt'))

    if download_link:
        # 獲取完整的下載URL
        download_url = 'https://www.gutenberg.org' + download_link['href']
        
        # 發送請求獲取書籍內容
        text_response = req.get(download_url)
        text_response.encoding = 'utf-8'
        
        # 提取書籍內容
        text_content = text_response.text
        if chinese_regex.search(text_content):
            # 儲存書籍內容到檔案
            file_name = re.sub(r'[\\/:*?"<>|]', '', book_name) + '.txt'
            with open(os.path.join('project_gutenberg', file_name), 'w', encoding='utf-8') as f:
                f.write(text_content)
            print(f"已保存: {file_name}")
            # 增加計數器
            saved_books_count += 1
        else:
            print(f"Skipped (no Chinese characters in content): {book_name}")
    else:
        print(f"No downloadable text file found for: {book_name}")

# 打印保存的書籍數量
print(f"總共保存了 {saved_books_count} 本書籍")


已保存: 豆棚閒話.txt
已保存: 戲中戲.txt
已保存: 比目魚.txt
已保存: 比目魚.txt
Skipped (non-Chinese book name): Study of Inner Cultivation
已保存: 三字經.txt
已保存: 山水情.txt
已保存: 山海經.txt
已保存: 施公案.txt
已保存: 施公案.txt
已保存: 易經.txt
已保存: 木蘭奇女傳.txt
已保存: 海公案.txt
已保存: 燕丹子.txt
已保存: 狄公案.txt
已保存: 百家姓.txt
已保存: 禮記.txt
已保存: 綠牡丹.txt
已保存: 詩經.txt
已保存: 麟兒報.txt
Skipped (non-Chinese book name): Study of Inner Cultivation
Yuan Yang Menghinese book name): Hu Die Mei
Skipped (non-Chinese book name): Qing Lou MengQi Hong Xiao Shi
已保存: 天豹圖.txt
已保存: 梁公九諫.txt
已保存: 長恨歌.txt
已保存: 李娃傳.txt
已保存: 玉樓春.txt
已保存: 漢書.txt
已保存: 引鳳蕭.txt
已保存: 今古奇觀.txt
已保存: 後西游記.txt
已保存: 飛跎全傳.txt
已保存: 佛說四十二章經.txt
已保存: 紅樓夢.txt
已保存: 洛神賦.txt
已保存: 晁氏儒言 一卷.txt
已保存: 水滸後傳.txt
已保存: 幼學瓊林.txt
已保存: 治世餘聞.txt
已保存: 琵琶記.txt
已保存: 雪月梅傳.txt
已保存: 龍川詞.txt
已保存: 三國志.txt
已保存: 隋唐演義.txt
已保存: 論語.txt
已保存: 滬語開路 = Conversational Exercises in the Shanghai Dialect.txt
已保存: 白圭志.txt
已保存: 孟子字義疏證.txt
已保存: 安樂集.txt
已保存: 鄧析子.txt
已保存: 醉醒石.txt
已保存: 唐鍾馗平鬼傳.txt
已保存: 春秋繁露.txt
已保存: 虬髯客傳.txt
已保存: 吳船錄.txt
已保存: 星槎勝覽.txt
已保存: 星槎勝

In [19]:
# 將下載下來的txt檔清理成只有中文字

# 資料夾路徑
folder_path = 'project_gutenberg'

# 中文正則表達式（含標點）
chinese_text_pattern = re.compile(r'[\u4e00-\u9fff，。？！：；「」『』（）《》、．‧·—…]+')

# 遍歷資料夾中的所有檔案
for filename in os.listdir(folder_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # 找出所有中文相關內容
        chinese_only = ''.join(chinese_text_pattern.findall(content))

        # 避免空檔案
        if chinese_only.strip():
            with open(file_path, 'w', encoding='utf-8') as file:
                file.write(chinese_only)
            print(f"已清理中文：{filename}")
        else:
            print(f"跳過（無中文）：{filename}")


已清理中文：海公案.txt
已清理中文：尉繚子.txt
已清理中文：三國志.txt
已清理中文：幻中游.txt
已清理中文：漱玉詞.txt
已清理中文：三略.txt
已清理中文：豔異編.txt
已清理中文：引鳳蕭.txt
已清理中文：詩品.txt
已清理中文：夢溪筆談, Volume 07-10.txt
已清理中文：徐霞客遊記.txt
已清理中文：申鑒.txt
已清理中文：辛棄疾詞選.txt
已清理中文：李太白集.txt
已清理中文：玉嬌梨.txt
已清理中文：朝花夕拾.txt
已清理中文：魏鄭公諫錄.txt
已清理中文：陶庵夢憶.txt
已清理中文：韓詩外傳.txt
已清理中文：穆天子传.txt
已清理中文：玉壺淸話.txt
已清理中文：阿Ｑ正傳.txt
已清理中文：賈誼新書.txt
已清理中文：道德經.txt
已清理中文：出師表.txt
已清理中文：粉妝樓全傳.txt
已清理中文：東坡樂府.txt
已清理中文：胡涂世界.txt
已清理中文：狐狸緣全傳.txt
已清理中文：闲情偶寄.txt
已清理中文：轟天雷.txt
已清理中文：滿江紅.txt
已清理中文：醉醒石.txt
已清理中文：梁公九諫.txt
已清理中文：虬髯客傳.txt
已清理中文：桯史.txt
已清理中文：滬語開路 = Conversational Exercises in the Shanghai Dialect.txt
已清理中文：方言.txt
已清理中文：楊家將演義.txt
已清理中文：東度記.txt
已清理中文：天工開物.txt
已清理中文：海上花列傳.txt
已清理中文：茶經.txt
已清理中文：洛神賦.txt
已清理中文：樂章集.txt
已清理中文：杜騙新書.txt
已清理中文：北夢瑣言.txt
已清理中文：商界現形記.txt
已清理中文：紅樓夢.txt
已清理中文：千字文.txt
已清理中文：抱朴子.txt
已清理中文：平妖傳.txt
已清理中文：珍珠舶.txt
已清理中文：玉雙魚.txt
已清理中文：吳越春秋.txt
已清理中文：綠牡丹.txt
已清理中文：合錦回文傳.txt
已清理中文：孫子兵法.txt
已清理中文：莊子的故事.txt
唐山過海的故事.txt
已清理中文：閒情偶寄.txt
已清理中文：琵琶記.txt
已清理中文：子不語.txt
已清理中文：快士傳.txt
已清理中文：